# Homework: Agentic RAG

**Module 1 - LLM Zoomcamp 2026**

This notebook implements an Agentic RAG system over the LLM Zoomcamp course lessons, covering document loading, indexing, retrieval, chunking, and agentic tool calling.

## Setup & Dependencies

In [2]:
# Fix BLAS threading to avoid memory issues on some systems
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

# Install extra homework dependencies if missing
import subprocess, sys

for package in ['gitsource', 'minsearch']:
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

print("Dependencies ready.")

Dependencies ready.


## Environment & Client Initialization

In [3]:
from dotenv import load_dotenv
load_dotenv()

import os
import json
from openai import OpenAI

# OpenRouter client
openai_client = OpenAI(
    base_url=os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    api_key=os.environ.get("OPENROUTER_API_KEY")
)

MODEL = "openai/gpt-oss-120b:free"

print("Environment loaded and client initialized.")

Environment loaded and client initialized.


## Q1: Load Lesson Documents

Fetch lesson markdown files from the course repository at commit `8c1834d`.

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

# Build knowledge base list
documents = []
for file in files:
    doc = file.parse()
    documents.append({
        "filename": doc["filename"],
        "content": doc["content"]
    })

print(f"Q1 Answer: {len(documents)} lesson pages")

Q1 Answer: 72 lesson pages


## Q2: Index and Search

Create a `minsearch` index with `content` as text field and `filename` as keyword field, then search for the agentic loop question.

In [5]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query=query, num_results=1)

first_filename = search_results[0]['filename']
print(f"Q2 Answer: {first_filename}")

Q2 Answer: 01-agentic-rag/lessons/14-agentic-loop.md


## Q3: RAG with Full Documents

Build a RAG pipeline, query the same question, and read the input token usage from the response.

In [6]:
# --- Modified RAG helper adapted for filename/content schema ---
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:
    def __init__(
        self,
        index,
        llm_client,
        model="openai/gpt-oss-120b:free"
    ):
        self.index = index
        self.llm_client = llm_client
        self.model = model

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append("---")
            lines.append(doc['filename'])
            lines.append(doc['content'])
        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return PROMPT_TEMPLATE.format(question=query, context=context)

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': INSTRUCTIONS},
            {'role': 'user', 'content': prompt}
        ]
        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )
        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response


# Initialize RAG with full-document index
rag_full = RAGBase(index=index, llm_client=openai_client, model=MODEL)

query = "How does the agentic loop keep calling the model until it stops?"
answer, response = rag_full.rag(query)
input_tokens_q3 = response.usage.input_tokens

print(f"Q3 Input tokens: {input_tokens_q3}")
print("\nAnswer preview:")
print(answer[:500])

Q3 Input tokens: 7177

Answer preview:
The agentic loop is just a **while‑True loop that repeats the same three steps**:

1. **Send the current message history to the model** (including any developer prompt, the user question, previous assistant messages, and any tool‑call results that have been added so far).  
2. **Inspect the model’s response**:  
   * If the response contains a `function_call` item, the code runs the requested tool (via the `make_call` helper), appends the tool’s output back into the `messages` list, and sets a f


## Q4: Chunking Documents

Split lesson pages into overlapping chunks of 2000 characters (step 1000) using `gitsource.chunk_documents`.

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Q4 Answer: {len(chunks)} chunks")

Q4 Answer: 295 chunks


## Q5: RAG with Chunked Documents

Re-index the chunks, run the same query through RAG, and compare input token count against Q3.

In [8]:
# Rebuild index on chunks
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename", "start"]
)
chunk_index.fit(chunks)

# RAG over chunks
rag_chunks = RAGBase(index=chunk_index, llm_client=openai_client, model=MODEL)

answer_c, response_c = rag_chunks.rag(query)
input_tokens_q5 = response_c.usage.input_tokens

print(f"Q5 Input tokens: {input_tokens_q5}")
print(f"Difference vs Q3: {input_tokens_q3 - input_tokens_q5} tokens ({input_tokens_q3/input_tokens_q5:.1f}x fewer)")

Q5 Input tokens: 2360
Difference vs Q3: 4817 tokens (3.0x fewer)


## Q6: Agentic RAG with Tool Calling

Give the LLM a `search` tool and let it decide when (and what) to search. We use a hand-written agent loop.

In [ ]:
# Define the search tool schema
course_tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Searches the chunked LLM Zoomcamp lesson documents to find relevant information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search keywords or concepts to locate within the lesson materials."
                    }
                },
                "required": ["query"]
            }
        }
    }
]


def search_tool(query: str) -> str:
    """Use this to search the course lesson chunks."""
    results = chunk_index.search(query, num_results=3)
    if not results:
        return "No matching course documents found for that query."
    return "\n\n".join([f"{r['filename']} (start={r.get('start', '?')})\n{r['content']}" for r in results])


# Agentic loop
target_question = "How does the agentic loop work, and how is it different from plain RAG?"

messages = [
    {
        "role": "system",
        "content": (
            "You are an autonomous course assistant for LLM Zoomcamp. "
            "You have access to a tool to search course lessons. Use it as many "
            "times as necessary to find exact details before answering. "
            "Make multiple searches with different keywords before answering."
        )
    },
    {"role": "user", "content": target_question}
]

import sys
sys.setrecursionlimit(2000)
max_turns = 10
search_calls = 0

for turn in range(max_turns):
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=course_tools,
        tool_choice="auto"
    )

    response_message = response.choices[0].message
    messages.append(response_message)

    if response_message.tool_calls:
        for tool_call in response_message.tool_calls:
            if tool_call.function.name == "search":
                search_calls += 1
                args = json.loads(tool_call.function.arguments)
                query_term = args.get("query", "")
                print(f"-> Tool call #{search_calls}: query = \"{query_term}\" ")
                tool_output = search_tool(query_term)
                if len(tool_output) > 15000:
                    tool_output = tool_output[:15000] + "\n... [Context Truncated] ..."
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": "search",
                    "content": tool_output
                })

-> Tool call #1: query = "agentic loop" 
-> Tool call #2: query = "plain RAG" 

[Loop Ended] Agent provided final answer.

Q6 Answer: 2 tool calls

Final agent answer:
### The agentic loop – what it is  

The **agentic loop** is the “while‑true” execution pattern that turns a language model into an *autonomous agent*.  

1. **Prompt → LLM** – The model receives a user goal (system + user messages).  
2. **Tool‑call check** – The model may return a **function‑call** (e.g., `search`, `calc`, `write_file`).  
3. **Execute the tool** – Your code runs the requested tool, captures its output, and formats it as a new “tool‑result” message.  
4. **Feed back** – The result message is appended to the conversation and the loop goes back to step 1.  
5. **Termination** – When the model replies with a normal message that **does not contain a tool call**, the loop stops and that final message is returned as the answer.

In pseudo‑code (the same code we wrote by hand in the course):

```python
messag

## Summary

In [10]:
print("=" * 50)
print("HOMEWORK ANSWERS SUMMARY")
print("=" * 50)
print(f"Q1: {len(documents)} lesson pages")
print(f"Q2: {first_filename}")
print(f"Q3: {input_tokens_q3} input tokens")
print(f"Q4: {len(chunks)} chunks")
print(f"Q5: {input_tokens_q5} input tokens ({input_tokens_q3/input_tokens_q5:.1f}x fewer)")
print(f"Q6: {search_calls} search tool calls")
print("=" * 50)

HOMEWORK ANSWERS SUMMARY
Q1: 72 lesson pages
Q2: 01-agentic-rag/lessons/14-agentic-loop.md
Q3: 7177 input tokens
Q4: 295 chunks
Q5: 2360 input tokens (3.0x fewer)
Q6: 2 search tool calls
